In [2]:
!python --version

Python 3.14.4


The system cannot find the path specified.


# ___HRNet___
---------------------

In [12]:
import os
import timeit
import argparse

import numpy as np
import torch
import torch.backends.cudnn as cudnn
import torch.nn as nn

In [4]:
print(torch.__version__)

2.11.0+cpu


In [5]:
# local imports

from lib.config import config, update_config
from lib.core.function import test, testval
from lib.utils.modelsummary import get_model_summary
from lib.utils.utils import create_logger

In [2]:
# from the docs;

# the pretrained weight was trained with the Cityscapes dataset
# models are trained and tested with the input size of 512x1024 and 1024x2048 respectively
# if multi-scale testing is used, we adopt scales: 0.5,0.75,1.0,1.25,1.5,1.75.

In [19]:
# the config files are in the experiments/ directory
os.listdir(r"./experiments/cityscapes/")

['seg_hrnet_ocr_w48_trainval_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484.yaml',
 'seg_hrnet_ocr_w48_train_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484.yaml',
 'seg_hrnet_ocr_w48_train_512x1024_sgd_lr1e-2_wd5e-4_bs_16_epoch484_paddle.yaml',
 'seg_hrnet_w48_trainval_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484x2.yaml',
 'seg_hrnet_w48_trainval_ohem_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484x2.yaml',
 'seg_hrnet_w48_train_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484.yaml',
 'seg_hrnet_w48_train_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484_paddle.yaml',
 'seg_hrnet_w48_train_512x1024_sgd_lr1e-2_wd5e-4_bs_16_epoch484_paddle.yaml',
 'seg_hrnet_w48_train_ohem_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484.yaml']

In [19]:
# the config we need is => HRNetV2-W48 + OCR

parser = argparse.ArgumentParser()

parser.add_argument("--cfg", help="experiment configure file name", type=str, default=r"./experiments/cityscapes/seg_hrnet_ocr_w48_train_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484.yaml")
parser.add_argument("DATASET.TEST_SET", default=R"data/list/cityscapes/test.lst", type=str)
parser.add_argument("TEST.MODEL_FILE", default=r"./../bin/hrnet_ocr_cs_trainval_8227_torch11.pth", type=str)
parser.add_argument("TEST.SCALE_LIST", default=(0.5,0.75,1.0,1.25,1.5,1.75), type=tuple[float])
parser.add_argument("TEST.FLIP_TEST", default=True, type=bool)

update_config(cfg=config, args=parser.get_default())

TypeError: _ActionsContainer.get_default() missing 1 required positional argument: 'dest'

In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description="Train segmentation network")

    parser.add_argument("--cfg", help="experiment configure file name", required=True, type=str)
    parser.add_argument("opts", help="Modify config options using the command-line", default=None, nargs=argparse.REMAINDER)

    args = parser.parse_args()
    update_config(config, args)

    return args


def main():
    args = parse_args()

    # cudnn related setting
    cudnn.benchmark = config.CUDNN.BENCHMARK
    cudnn.deterministic = config.CUDNN.DETERMINISTIC
    cudnn.enabled = config.CUDNN.ENABLED

    # build model
    if torch.__version__.startswith("1"):
        module = eval("models." + config.MODEL.NAME)
        module.BatchNorm2d_class = module.BatchNorm2d = torch.nn.BatchNorm2d
        
    model = eval("models." + config.MODEL.NAME + ".get_seg_model")(config)

    dump_input = torch.rand((1, 3, config.TRAIN.IMAGE_SIZE[1], config.TRAIN.IMAGE_SIZE[0]))
    logger.info(get_model_summary(model.cuda(), dump_input.cuda()))

    if config.TEST.MODEL_FILE:
        model_state_file = config.TEST.MODEL_FILE
    else:
        model_state_file = os.path.join(final_output_dir, "final_state.pth")
    logger.info("=> loading model from {}".format(model_state_file))

    pretrained_dict = torch.load(model_state_file)
    if "state_dict" in pretrained_dict:
        pretrained_dict = pretrained_dict["state_dict"]
    model_dict = model.state_dict()
    pretrained_dict = {k[6:]: v for k, v in pretrained_dict.items() if k[6:] in model_dict.keys()}
    for k, _ in pretrained_dict.items():
        logger.info("=> loading {} from pretrained model".format(k))
    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)

    gpus = list(config.GPUS)
    model = nn.DataParallel(model, device_ids=gpus).cuda()

    # prepare data
    test_size = (config.TEST.IMAGE_SIZE[1], config.TEST.IMAGE_SIZE[0])
    test_dataset = eval("datasets." + config.DATASET.DATASET)(
        root=config.DATASET.ROOT,
        list_path=config.DATASET.TEST_SET,
        num_samples=None,
        num_classes=config.DATASET.NUM_CLASSES,
        multi_scale=False,
        flip=False,
        ignore_label=config.TRAIN.IGNORE_LABEL,
        base_size=config.TEST.BASE_SIZE,
        crop_size=test_size,
        downsample_rate=1,
    )

    testloader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=config.WORKERS, pin_memory=True)

    start = timeit.default_timer()
    if "val" in config.DATASET.TEST_SET:
        mean_IoU, IoU_array, pixel_acc, mean_acc = testval(config, test_dataset, testloader, model)

        msg = "MeanIU: {: 4.4f}, Pixel_Acc: {: 4.4f}, \
            Mean_Acc: {: 4.4f}, Class IoU: ".format(mean_IoU, pixel_acc, mean_acc)
        logging.info(msg)
        logging.info(IoU_array)
    elif "test" in config.DATASET.TEST_SET:
        test(config, test_dataset, testloader, model, sv_dir=final_output_dir)

    end = timeit.default_timer()
    logger.info("Mins: %d" % np.int((end - start) / 60))
    logger.info("Done")
